# Step 0 — what data exists, for which date range, per source

Simonyi time-loss investigation (`simonyi_time_loss/`), handoff Section 9 guardrail: *report Step 0 before any analysis*.
Range: `day_obs` 20251025 → 20260714 (263 calendar nights). Every table below is written to `../data/step0/` so the
README and `ai_notes.md` can cite it. Nothing here is a result; it is an inventory.

Sources probed:
1. **ConsDB** `cdb_lsstcam.visit1` — visits per night (any row / on-sky `airmass > 0` / standard on-sky image types / on-sky `engtest`).
2. **Observatory state** `lsst.sal.Scheduler.logevent_observatoryStatus` via `rubin_nights.observatory_status` — WEATHER / FAULT / DOWNTIME / OPERATIONAL / IDLE / UNKNOWN hours per night.
3. **Narrative log** (`narrativelog/messages`, `time_lost > 0`) — hours per night and `time_lost_type`.
4. **EFD weather telemetry** for the Section 5 criteria — weather-tower ESS:301 (dew point, temperature, RH, wind), camera inlet ESS:111 dew point, `MTCamera.utiltrunk_UT.coolPipeSplyTemp` (Dynalene supply), `MTDome.apertureShutter` (dome open/closed).
5. **Ephemeris** — −18° and −12° night limits per `day_obs`.

Runs from `notebooks/` on an SDF login node with `source /sdf/group/rubin/sw/loadLSST.bash; setup lsst_distrib`
(gives `rubin_nights` 0.18.2). ConsDB falls back to the USDF Postgres replica; the narrative log needs the RSP token in `~/.token_file`.

In [ ]:
import sys, logging, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from time_loss import DAY_OBS_MIN, DAY_OBS_MAX, access, nights

pd.set_option("display.width", 220); pd.set_option("display.max_rows", 300)
OUT = Path("../data/step0"); OUT.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.WARNING)
for name in ("httpx", "numexpr", "time_loss.access"):
    logging.getLogger(name).setLevel(logging.WARNING)
print(DAY_OBS_MIN, DAY_OBS_MAX)

## 5. Ephemeris: night limits (no data, no judgement)

In [ ]:
limits = nights.night_limits(DAY_OBS_MIN, DAY_OBS_MAX)
limits.to_csv(OUT / "night_limits.csv", index=False)
print(len(limits), "nights")
print(limits[["night_hours_18", "night_hours_12"]].agg(["min", "max", "sum"]).round(1))
limits.head(3)

## 1. ConsDB `visit1` coverage per night

In [ ]:
consdb = access.get_consdb_client()
sql = f"""SELECT day_obs, count(*) AS n_all,
 sum(CASE WHEN airmass>0 THEN 1 ELSE 0 END) AS n_onsky,
 sum(CASE WHEN airmass>0 AND img_type IN ('acq','science','cwfs','focus') THEN 1 ELSE 0 END) AS n_onsky_std,
 sum(CASE WHEN airmass>0 AND img_type='engtest' THEN 1 ELSE 0 END) AS n_engtest_onsky,
 min(obs_start_mjd) AS mjd_first, max(obs_end_mjd) AS mjd_last
 FROM cdb_lsstcam.visit1 WHERE day_obs BETWEEN {DAY_OBS_MIN} AND {DAY_OBS_MAX}
 GROUP BY day_obs ORDER BY day_obs"""
cov = consdb.query(sql)
cov.to_csv(OUT / "consdb_visit1_coverage.csv", index=False)
cal = limits[["day_obs"]].merge(cov, on="day_obs", how="left")
for c in ["n_all", "n_onsky", "n_onsky_std", "n_engtest_onsky"]:
    cal[c] = cal[c].fillna(0).astype(int)
cal.to_csv(OUT / "calendar_coverage.csv", index=False)
print("calendar nights", len(cal), "| any visit1 row", (cal.n_all > 0).sum(),
      "| any on-sky (airmass>0)", (cal.n_onsky > 0).sum(), "| >= 20 on-sky", (cal.n_onsky >= 20).sum())
print("on-sky visits total", cal.n_onsky.sum(), "| of which engtest", cal.n_engtest_onsky.sum())
imgt = consdb.query(f"SELECT img_type, (airmass>0) AS onsky, count(*) AS n FROM cdb_lsstcam.visit1 "
                    f"WHERE day_obs BETWEEN {DAY_OBS_MIN} AND {DAY_OBS_MAX} GROUP BY img_type, (airmass>0) ORDER BY n DESC")
imgt

In [ ]:
zero = cal.query("n_onsky == 0").day_obs.tolist()
print(len(zero), "nights with ZERO on-sky visits")
# contiguous runs
runs, start, prev = [], None, None
for d, z in zip(cal.day_obs, cal.n_onsky == 0):
    if z and start is None: start = d
    if not z and start is not None: runs.append((start, prev)); start = None
    prev = d
if start is not None: runs.append((start, prev))
runs

## 2. Observatory state (WEATHER / FAULT / DOWNTIME ...) — `rubin_nights.observatory_status`

In [ ]:
import rubin_nights.dayobs_utils as rn
from rubin_nights import observatory_status as obs
from astropy.time import TimeDelta
efd = access.get_efd_client()
t0 = rn.day_obs_to_time(DAY_OBS_MIN); t1 = rn.day_obs_to_time(DAY_OBS_MAX) + TimeDelta(1, format="jd")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    periods = obs.get_observatory_state_times(t0, t1, efd)
periods["day_obs"] = periods.day_obs.astype(int)
periods.to_parquet(OUT / "obs_state_periods.parquet")
print("periods:", len(periods))
print(periods.groupby("type").agg(n=("hours", "size"), hours=("hours", "sum"), first=("start", "min"), last=("end", "max")).round(1))
per_night = periods.groupby(["day_obs", "type"]).hours.sum().unstack(fill_value=0.0).round(2)
per_night["excl_sum"] = per_night[[c for c in ["OPERATIONAL", "FAULT", "IDLE", "UNKNOWN"] if c in per_night]].sum(axis=1)
per_night["night_hours_12"] = limits.set_index("day_obs").night_hours_12.reindex(per_night.index)
per_night["coverage"] = (per_night.excl_sum / per_night.night_hours_12).round(2)
per_night.to_csv(OUT / "obs_state_per_night.csv")
first = per_night.index.min()
missing = [d for d in cal.day_obs if d >= first and d not in per_night.index]
print("first night with a state record:", first, "| nights since then with no record:", missing)
print("nights with exclusive-state coverage < 0.8 of the -12 night:")
per_night.query("coverage < 0.8")

In [ ]:
w = per_night.get("WEATHER", pd.Series(dtype=float))
print("WEATHER nights:", (w > 0).sum(), "| hours:", round(w.sum(), 1), "| nights with WEATHER >= 6 h:", (w >= 6).sum())
w[w > 0]

## 3. Narrative log — entries with `time_lost > 0`

`day_obs` is derived from `date_begin` with the 12:00 UTC rollover (the API has no day_obs field). AuxTel-component entries are dropped for the Simonyi table.

In [ ]:
nl = access.get_endpoints()["narrative_log"]
logs = nl.query_log(t0, t1, {"min_time_lost": "0.00001"})
print("rows:", len(logs))
logs["day_obs"] = (pd.to_datetime(logs["date_begin"], format="ISO8601") - pd.Timedelta(hours=12)).dt.strftime("%Y%m%d").astype(int)
logs.to_parquet(OUT / "narrative_log_timelost.parquet")
print(logs.time_lost_type.value_counts().to_dict())
print(logs.component.astype(str).value_counts().to_dict())
sim = logs[~logs.component.astype(str).str.contains("AuxTel", case=False)]
pn = sim.groupby(["day_obs", "time_lost_type"]).time_lost.sum().unstack(fill_value=0.0).round(2)
pn.to_csv(OUT / "narrative_log_timelost_per_night.csv")
print("non-AuxTel: nights with weather loss", (pn.get("weather", 0) > 0).sum(), "| with fault loss", (pn.get("fault", 0) > 0).sum())
sim.groupby([sim.day_obs // 100, "time_lost_type"]).time_lost.sum().unstack(fill_value=0).round(1)

## 4. EFD weather telemetry for the Section 5 criteria

Sensor inventory (which `salIndex` / `sensorName` / `location` publish on five sample days), then weekly sample counts of the fields the criteria need, then the dome shutter.

In [ ]:
def q(s):
    try:
        return efd.query(s)
    except Exception as e:  # noqa: BLE001
        print("ERR", type(e).__name__, s[:100]); return pd.DataFrame()

rows = []
for topic in ["lsst.sal.ESS.dewPoint", "lsst.sal.ESS.temperature", "lsst.sal.ESS.relativeHumidity", "lsst.sal.ESS.airFlow", "lsst.sal.ESS.rainRate"]:
    for day in ["2025-11-15", "2026-01-15", "2026-03-15", "2026-05-15", "2026-07-10"]:
        d = q(f'SELECT "salIndex","sensorName","location" FROM "{topic}" WHERE time >= \'{day}T00:00:00Z\' AND time < \'{day}T00:20:00Z\'')
        if len(d) == 0:
            continue
        for (si, sn, loc), n in d.groupby(["salIndex", "sensorName", "location"]).size().items():
            rows.append(dict(topic=topic.split(".")[-1], day=day, salIndex=si, sensorName=sn, location=loc[:60], n_20min=n))
inv = pd.DataFrame(rows); inv.to_csv(OUT / "ess_sensor_inventory.csv", index=False)
inv.query("topic != 'temperature' or salIndex in [301, 111]").pivot_table(index=["topic", "salIndex", "sensorName", "location"], columns="day", values="n_20min", aggfunc="sum")

In [ ]:
rng = f"time >= '{t0.utc.isot}Z' AND time < '{t1.utc.isot}Z'"
fields = [("lsst.sal.ESS.dewPoint", "dewPointItem", 301), ("lsst.sal.ESS.dewPoint", "dewPointItem", 111),
          ("lsst.sal.ESS.temperature", "temperatureItem0", 301), ("lsst.sal.ESS.relativeHumidity", "relativeHumidityItem", 301),
          ("lsst.sal.ESS.airFlow", "speed", 301), ("lsst.sal.ESS.airFlow", "maxSpeed", 301)]
weekly = {}
for topic, fld, idx in fields:
    d = q(f'SELECT count("{fld}") AS n FROM "{topic}" WHERE {rng} AND "salIndex"={idx} GROUP BY time(7d)')
    if len(d): weekly[f"{topic.split('.')[-1]}[{idx}].{fld}"] = d.n
d = q(f'SELECT count("coolPipeSplyTemp") AS n FROM "lsst.sal.MTCamera.utiltrunk_UT" WHERE {rng} GROUP BY time(7d)')
if len(d): weekly["utiltrunk_UT.coolPipeSplyTemp"] = d.n
weekly = pd.DataFrame(weekly); weekly.to_csv(OUT / "ess_weekly_counts.csv")
print("weeks with any zero-count field:", weekly.index[(weekly == 0).any(axis=1)].tolist())
weekly.describe().round(0)

In [ ]:
sh = q(f'SELECT max("positionActual0") AS smax, mean("positionActual0") AS smean FROM "lsst.sal.MTDome.apertureShutter" WHERE {rng} GROUP BY time(1d,12h)')
sh.index = sh.index.tz_convert(None); sh["day_obs"] = sh.index.strftime("%Y%m%d").astype(int)
sh = sh.query("day_obs >= @DAY_OBS_MIN and day_obs <= @DAY_OBS_MAX")
sh.to_csv(OUT / "shutter_daily.csv", index=False)
print("nights:", len(sh), "| no shutter telemetry:", sh.smax.isna().sum(), "| shutter max > 50 %:", (sh.smax > 50).sum())
print("never > 50 %:", sh.query("smax.isna() or smax <= 50").day_obs.tolist())

## Zero-visit nights cross-matched

For each of the nights with no on-sky visit: shutter state, state bits, narrative-log hours, and the Source 1 / Source 2 status where the cross-check lists a disagreement (nights that agree are not in that file).

In [ ]:
cc = {}
for line in open("../data/raw/crosscheck_S1_vs_S2_2025-10-25_to_2026-07-14.md", encoding="utf-8"):
    if line.startswith("| 20") and line.count("|") >= 11:
        c = [x.strip() for x in line.strip().strip("|").split("|")]
        cc[int(c[0].replace("-", ""))] = (c[2], c[3])
shi = sh.set_index("day_obs")
z = pd.DataFrame({
    "day_obs": zero,
    "visit1_rows": [int(cal.set_index("day_obs").loc[d, "n_all"]) for d in zero],
    "shutter_max": [shi.smax.get(d, np.nan) for d in zero],
    "WEATHER_h": [per_night.get("WEATHER", pd.Series(dtype=float)).get(d, np.nan) for d in zero],
    "DOWNTIME_h": [per_night.get("DOWNTIME", pd.Series(dtype=float)).get(d, np.nan) for d in zero],
    "FAULT_h": [per_night.get("FAULT", pd.Series(dtype=float)).get(d, np.nan) for d in zero],
    "nl_weather_h": [pn.get("weather", pd.Series(dtype=float)).get(d, 0.0) for d in zero],
    "nl_fault_h": [pn.get("fault", pd.Series(dtype=float)).get(d, 0.0) for d in zero],
    "S1": [cc.get(d, ("(agree / not listed)", ""))[0] for d in zero],
    "S2": [cc.get(d, ("", ""))[1] for d in zero],
}).round(2)
z.to_csv(OUT / "zero_visit_nights.csv", index=False)
print("zero-visit nights:", len(z), "| shutter opened > 50 %:", (z.shutter_max > 50).sum(),
      "| WEATHER >= 6 h:", (z.WEATHER_h >= 6).sum(), "| DOWNTIME >= 6 h:", (z.DOWNTIME_h >= 6).sum())
z